In [2]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

Current working directory:
/Users/nimo/crypto-statistical-arbitrage


# 01 Data Collection

This notebook downloads hourly Binance Spot OHLCV data for ten liquid cryptocurrency pairs from 2023-01-01 to 2025-12-31.

In [49]:
from pathlib import Path
from io import BytesIO
import zipfile
import requests
import pandas as pd
from tqdm.auto import tqdm

In [51]:
from pathlib import Path

symbols = [
    "BTCUSDT",
    "ETHUSDT",
    "BNBUSDT",
    "XRPUSDT",
    "ADAUSDT",
    "DOGEUSDT",
    "SOLUSDT",
    "LTCUSDT",
    "LINKUSDT",
    "AVAXUSDT",
]

start_date = "2023-01-01"
end_date = "2025-12-31"
interval = "1h"

raw_data_dir = Path("data/raw")
raw_data_dir.mkdir(parents=True, exist_ok=True)

print("Raw data directory:")
print(raw_data_dir.resolve())

Raw data directory:
/Users/nimo/crypto-statistical-arbitrage/data/raw


In [53]:
import pandas as pd

months = pd.period_range(
    start=start_date,
    end=end_date,
    freq="M",
)

print("Number of months:", len(months))
print(months)

Number of months: 36
PeriodIndex(['2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06',
             '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12',
             '2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06',
             '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12',
             '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06',
             '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12'],
            dtype='period[M]')


In [55]:
symbol = "BTCUSDT"
month = "2023-01"

url = (
    f"https://data.binance.vision/data/spot/monthly/klines/"
    f"{symbol}/{interval}/{symbol}-{interval}-{month}.zip"
)

print(url)

https://data.binance.vision/data/spot/monthly/klines/BTCUSDT/1h/BTCUSDT-1h-2023-01.zip


In [57]:
response = requests.get(url, timeout=60)
response.raise_for_status()

print("Download successful")
print("Status code:", response.status_code)
print("File size:", len(response.content), "bytes")

Download successful
Status code: 200
File size: 44765 bytes


In [59]:
columns = [
    "open_time",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "close_time",
    "quote_asset_volume",
    "number_of_trades",
    "taker_buy_base_volume",
    "taker_buy_quote_volume",
    "ignore",
]

with zipfile.ZipFile(BytesIO(response.content)) as zipped_file:
    print("Files inside ZIP:", zipped_file.namelist())

    csv_filename = zipped_file.namelist()[0]

    with zipped_file.open(csv_filename) as csv_file:
        test_df = pd.read_csv(
            csv_file,
            header=None,
            names=columns,
        )

test_df.head()

Files inside ZIP: ['BTCUSDT-1h-2023-01.csv']


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume,ignore
0,1672531200000,16541.77,16545.70,16508.39,16529.67,4364.83570,1672534799999,7.214629e+07,149854,2179.94772,3.603235e+07,0
1,1672534800000,16529.59,16556.80,16525.78,16551.47,3590.06669,1672538399999,5.937676e+07,126556,1730.24901,2.861742e+07,0
2,1672538400000,16551.47,16559.77,16538.14,16548.19,3318.84038,1672541999999,5.491945e+07,115398,1611.12302,2.666087e+07,0
3,1672542000000,16548.19,16548.19,16518.21,16533.04,4242.08050,1672545599999,7.012254e+07,137724,2096.09287,3.464904e+07,0
4,1672545600000,16533.04,16535.97,16511.92,16521.85,4285.00909,1672549199999,7.080264e+07,129535,2188.40175,3.615982e+07,0


In [61]:
print("Shape:", test_df.shape)

Shape: (744, 12)


In [63]:
test_df["open_time"] = pd.to_datetime(
    test_df["open_time"],
    unit="ms",
    utc=True,
)

test_df["close_time"] = pd.to_datetime(
    test_df["close_time"],
    unit="ms",
    utc=True,
)

test_df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume,ignore
0,2023-01-01 00:00:00+00:00,16541.77,16545.70,16508.39,16529.67,4364.83570,2023-01-01 00:59:59.999000+00:00,7.214629e+07,149854,2179.94772,3.603235e+07,0
1,2023-01-01 01:00:00+00:00,16529.59,16556.80,16525.78,16551.47,3590.06669,2023-01-01 01:59:59.999000+00:00,5.937676e+07,126556,1730.24901,2.861742e+07,0
2,2023-01-01 02:00:00+00:00,16551.47,16559.77,16538.14,16548.19,3318.84038,2023-01-01 02:59:59.999000+00:00,5.491945e+07,115398,1611.12302,2.666087e+07,0
3,2023-01-01 03:00:00+00:00,16548.19,16548.19,16518.21,16533.04,4242.08050,2023-01-01 03:59:59.999000+00:00,7.012254e+07,137724,2096.09287,3.464904e+07,0
4,2023-01-01 04:00:00+00:00,16533.04,16535.97,16511.92,16521.85,4285.00909,2023-01-01 04:59:59.999000+00:00,7.080264e+07,129535,2188.40175,3.615982e+07,0


In [65]:
numeric_columns = [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "quote_asset_volume",
    "number_of_trades",
    "taker_buy_base_volume",
    "taker_buy_quote_volume",
]

test_df[numeric_columns] = test_df[numeric_columns].apply(
    pd.to_numeric,
    errors="coerce",
)

test_df["symbol"] = symbol

In [67]:
print(test_df.dtypes)
print(test_df["open_time"].min())
print(test_df["open_time"].max())
print(test_df.isna().sum())

open_time                 datetime64[ns, UTC]
open                                  float64
high                                  float64
low                                   float64
close                                 float64
volume                                float64
close_time                datetime64[ns, UTC]
quote_asset_volume                    float64
number_of_trades                        int64
taker_buy_base_volume                 float64
taker_buy_quote_volume                float64
ignore                                  int64
symbol                                 object
dtype: object
2023-01-01 00:00:00+00:00
2023-01-31 23:00:00+00:00
open_time                 0
open                      0
high                      0
low                       0
close                     0
volume                    0
close_time                0
quote_asset_volume        0
number_of_trades          0
taker_buy_base_volume     0
taker_buy_quote_volume    0
ignore                    0


## Download Functions

The following function downloads and processes one monthly Binance Kline file.

In [69]:
def download_monthly_klines(symbol, interval, month):
    """
    Download and process one monthly Binance Spot Kline file.

    Parameters
    ----------
    symbol : str
        Trading pair, for example "BTCUSDT".
    interval : str
        Kline interval, for example "1h".
    month : str
        Month in YYYY-MM format, for example "2023-01".

    Returns
    -------
    pandas.DataFrame
        Processed hourly OHLCV data for the selected month.
    """

    url = (
        "https://data.binance.vision/data/spot/monthly/klines/"
        f"{symbol}/{interval}/{symbol}-{interval}-{month}.zip"
    )

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    columns = [
        "open_time",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "close_time",
        "quote_asset_volume",
        "number_of_trades",
        "taker_buy_base_volume",
        "taker_buy_quote_volume",
        "ignore",
    ]

    with zipfile.ZipFile(BytesIO(response.content)) as zipped_file:
        csv_filename = zipped_file.namelist()[0]

        with zipped_file.open(csv_filename) as csv_file:
            df = pd.read_csv(
                csv_file,
                header=None,
                names=columns,
            )

    # Convert timestamps to UTC datetime
    df["open_time"] = convert_binance_timestamp(
        df["open_time"]
    )

    df["close_time"] = convert_binance_timestamp(
        df["close_time"]
    )

    # Convert market-data columns to numeric values
    numeric_columns = [
        "open",
        "high",
        "low",
        "close",
        "volume",
        "quote_asset_volume",
        "number_of_trades",
        "taker_buy_base_volume",
        "taker_buy_quote_volume",
    ]

    df[numeric_columns] = df[numeric_columns].apply(
        pd.to_numeric,
        errors="coerce",
    )

    df["symbol"] = symbol

    return df

In [71]:
def convert_binance_timestamp(series):
    """
    Convert Binance timestamps to UTC datetime.

    Binance Spot timestamps before 2025 are generally in milliseconds,
    while timestamps from 2025 onwards are in microseconds.
    """

    numeric_series = pd.to_numeric(series, errors="coerce")

    # Millisecond timestamps are approximately 13 digits.
    # Microsecond timestamps are approximately 16 digits.
    unit = "us" if numeric_series.dropna().median() > 10**14 else "ms"

    return pd.to_datetime(
        numeric_series,
        unit=unit,
        utc=True,
        errors="coerce",
    )

In [73]:
test_download = download_monthly_klines(
    symbol="ETHUSDT",
    interval="1h",
    month="2023-02",
)

print("Shape:", test_download.shape)
print("Start:", test_download["open_time"].min())
print("End:", test_download["open_time"].max())

test_download.head()

Shape: (672, 13)
Start: 2023-02-01 00:00:00+00:00
End: 2023-02-28 23:00:00+00:00


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume,ignore,symbol
0,2023-02-01 00:00:00+00:00,1585.32,1590.00,1576.43,1583.34,16798.7557,2023-02-01 00:59:59.999000+00:00,2.660024e+07,30053,8777.1116,1.389975e+07,0,ETHUSDT
1,2023-02-01 01:00:00+00:00,1583.34,1592.94,1580.35,1588.92,14006.6021,2023-02-01 01:59:59.999000+00:00,2.221817e+07,21616,7417.9195,1.176849e+07,0,ETHUSDT
2,2023-02-01 02:00:00+00:00,1588.93,1589.44,1582.21,1585.66,8378.3636,2023-02-01 02:59:59.999000+00:00,1.327820e+07,14928,4473.0491,7.088807e+06,0,ETHUSDT
3,2023-02-01 03:00:00+00:00,1585.66,1585.98,1578.80,1584.61,7889.1912,2023-02-01 03:59:59.999000+00:00,1.248484e+07,14224,4170.8841,6.600422e+06,0,ETHUSDT
4,2023-02-01 04:00:00+00:00,1584.62,1585.65,1581.32,1582.98,6762.6261,2023-02-01 04:59:59.999000+00:00,1.071202e+07,13202,3072.3901,4.866278e+06,0,ETHUSDT


In [74]:
def download_symbol_history(symbol, interval, months):
    """
    Download and combine all monthly files for one symbol.
    """

    monthly_data = []
    failed_months = []

    for month in tqdm(months, desc=f"Downloading {symbol}"):
        month_string = str(month)

        try:
            month_df = download_monthly_klines(
                symbol=symbol,
                interval=interval,
                month=month_string,
            )

            monthly_data.append(month_df)

        except requests.RequestException as error:
            print(f"Failed: {symbol}, {month_string}: {error}")
            failed_months.append(month_string)

        except zipfile.BadZipFile:
            print(f"Invalid ZIP: {symbol}, {month_string}")
            failed_months.append(month_string)

    if not monthly_data:
        raise ValueError(f"No data were downloaded for {symbol}.")

    symbol_df = pd.concat(
        monthly_data,
        ignore_index=True,
    )

    symbol_df = symbol_df.sort_values("open_time")
    symbol_df = symbol_df.drop_duplicates(
        subset=["symbol", "open_time"],
        keep="first",
    )

    return symbol_df, failed_months

In [77]:
btc_df, btc_failed_months = download_symbol_history(
    symbol="BTCUSDT",
    interval=interval,
    months=months,
)

print("BTC shape:", btc_df.shape)
print("Start:", btc_df["open_time"].min())
print("End:", btc_df["open_time"].max())
print("Failed months:", btc_failed_months)

BTC shape: (26303, 13)
Start: 2023-01-01 00:00:00+00:00
End: 2025-12-31 23:00:00+00:00
Failed months: []


In [79]:
expected_hours = pd.date_range(
    start="2023-01-01 00:00:00",
    end="2025-12-31 23:00:00",
    freq="1h",
    tz="UTC",
)

actual_hours = pd.DatetimeIndex(
    btc_df["open_time"].dropna().unique()
)

missing_hours = expected_hours.difference(actual_hours)

print("Expected hours:", len(expected_hours))
print("Actual unique hours:", len(actual_hours))
print("Missing hours:", len(missing_hours))
print(missing_hours)

Expected hours: 26304
Actual unique hours: 26303
Missing hours: 1
DatetimeIndex(['2023-03-24 13:00:00+00:00'], dtype='datetime64[ns, UTC]', freq='h')


In [81]:
btc_df.loc[
    (btc_df["open_time"] >= "2023-03-24 10:00:00+00:00")
    & (btc_df["open_time"] <= "2023-03-24 16:00:00+00:00"),
    ["open_time", "open", "high", "low", "close", "volume"]
]

,open_time,open,high,low,close,volume
1978,2023-03-24 10:00:00+00:00,28041.11,28085.06,27941.00,28039.71,3639.24610
1979,2023-03-24 11:00:00+00:00,28039.71,28091.03,27963.84,28080.00,1267.41714
1980,2023-03-24 12:00:00+00:00,28080.00,28080.00,28080.00,28080.00,0.00000
1981,2023-03-24 14:00:00+00:00,28079.99,28253.01,27835.00,27989.06,8983.24018
1982,2023-03-24 15:00:00+00:00,27989.07,28076.82,27843.41,28018.04,5198.28681
1983,2023-03-24 16:00:00+00:00,28018.04,28059.63,27831.33,27831.33,3377.68679


### BTCUSDT Data Quality Note

One missing hourly observation was identified:

- Missing timestamp: `2023-03-24 13:00:00 UTC`

The preceding hour (`12:00 UTC`) recorded zero volume and no price movement, suggesting a possible exchange interruption or maintenance period. The missing observation is left unfilled to avoid introducing synthetic prices.

In [84]:
btc_output_path = (
    raw_data_dir / "BTCUSDT_1h_2023_2025.parquet"
)

btc_df.to_parquet(
    btc_output_path,
    index=False,
)

print("Saved to:")
print(btc_output_path.resolve())

Saved to:
/Users/nimo/crypto-statistical-arbitrage/data/raw/BTCUSDT_1h_2023_2025.parquet


## Download Full Cryptocurrency Universe

The following section downloads and saves the full 2023–2025 hourly history for all selected cryptocurrency pairs.

In [89]:
# Exclude BTCUSDT because it has already been downloaded and saved
remaining_symbols = [
    symbol for symbol in symbols
    if symbol != "BTCUSDT"
]


all_data = {}
download_summary = []

for symbol in symbols:
    print(f"\nStarting {symbol}")

    symbol_df, failed_months = download_symbol_history(
        symbol=symbol,
        interval=interval,
        months=months,
    )

    output_path = (
        raw_data_dir / f"{symbol}_{interval}_2023_2025.parquet"
    )

    symbol_df.to_parquet(
        output_path,
        index=False,
    )

    all_data[symbol] = symbol_df

    download_summary.append(
        {
            "symbol": symbol,
            "rows": len(symbol_df),
            "start": symbol_df["open_time"].min(),
            "end": symbol_df["open_time"].max(),
            "failed_months": len(failed_months),
            "output_file": str(output_path),
        }
    )

    print(f"Saved {symbol}: {len(symbol_df):,} rows")
    print(f"Failed months: {failed_months}")


Starting BTCUSDT


Saved BTCUSDT: 26,303 rows
Failed months: []

Starting ETHUSDT


Saved ETHUSDT: 26,303 rows
Failed months: []

Starting BNBUSDT


Saved BNBUSDT: 26,303 rows
Failed months: []

Starting XRPUSDT


Saved XRPUSDT: 26,303 rows
Failed months: []

Starting ADAUSDT


...

Saved DOGEUSDT: 26,303 rows
Failed months: []

Starting SOLUSDT


Saved SOLUSDT: 26,303 rows
Failed months: []

Starting LTCUSDT


Saved LTCUSDT: 26,303 rows
Failed months: []

Starting LINKUSDT


Saved LINKUSDT: 26,303 rows
Failed months: []

Starting AVAXUSDT


Saved AVAXUSDT: 26,303 rows
Failed months: []


In [90]:
download_summary_df = pd.DataFrame(download_summary)

download_summary_df

,symbol,rows,start,end,failed_months,output_file
0,BTCUSDT,26303,2023-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,0,data/raw/BTCUSDT_1h_2023_2025.parquet
1,ETHUSDT,26303,2023-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,0,data/raw/ETHUSDT_1h_2023_2025.parquet
2,BNBUSDT,26303,2023-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,0,data/raw/BNBUSDT_1h_2023_2025.parquet
3,XRPUSDT,26303,2023-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,0,data/raw/XRPUSDT_1h_2023_2025.parquet
4,ADAUSDT,26303,2023-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,0,data/raw/ADAUSDT_1h_2023_2025.parquet
5,DOGEUSDT,26303,2023-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,0,data/raw/DOGEUSDT_1h_2023_2025.parquet
6,SOLUSDT,26303,2023-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,0,data/raw/SOLUSDT_1h_2023_2025.parquet
7,LTCUSDT,26303,2023-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,0,data/raw/LTCUSDT_1h_2023_2025.parquet
8,LINKUSDT,26303,2023-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,0,data/raw/LINKUSDT_1h_2023_2025.parquet
9,AVAXUSDT,26303,2023-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,0,data/raw/AVAXUSDT_1h_2023_2025.parquet


In [93]:
summary_path = raw_data_dir / "download_summary_2023_2025.csv"

download_summary_df.to_csv(
    summary_path,
    index=False
)

print("Summary saved to:")
print(summary_path.resolve())

Summary saved to:
/Users/nimo/crypto-statistical-arbitrage/data/raw/download_summary_2023_2025.csv


## Data Collection Summary

Hourly Binance Spot OHLCV data were successfully downloaded for 10 liquid USDT cryptocurrency pairs from January 2023 to December 2025.

Each dataset contains 26,303 hourly observations. No monthly downloads failed. A common missing hourly timestamp will be investigated in the data-quality stage.